In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.ipynb").is_file())
DATA = ROOT / "data"
OUT = ROOT / "output"
COUNTY_FILE = r"Z:\Boundaries\Census_Boundary\County\US_County.shp"
STATE_FILE = r"Z:\Boundaries\Census_Boundary\State\US_State.shp"

CFG = {'features': ['LONGITUDE',
              'LATITUDE',
              'HDD',
              'CDD',
              'surface_net_thermal_radiation_sum',
              'wind_speed_10m',
              'PM25',
              'NTL',
              'NDBI',
              'EVI',
              'ACS_PCT_BLACK_NONHISP',
              'ACS_PCT_HISPANIC',
              'ACS_PCT_AGE_ABOVE65',
              'ACS_PCT_ENGL_NOT_ALL',
              'ACS_PCT_PERSON_INC_BELOW99',
              'ACS_PCT_UNEMPLOY',
              'AC_RECS',
              'ACS_PCT_HU_ELEC',
              'ACS_PCT_HU_SOLAR',
              'ACS_PCT_OWNER_HU_COST_30PCT',
              'ACS_PCT_RENTER_HU',
              'ACS_PCT_1UP_PERS_1ROOM',
              'ACS_MEDIAN_HOME_VALUE',
              'ACS_MEDIAN_YEAR_BUILT',
              'PCT_DEM',
              'RH_mean'],
 'labels': {'LONGITUDE': 'Longitude',
            'LATITUDE': 'Latitude',
            'HDD': 'Heating degree days',
            'CDD': 'Cooling degree days',
            'surface_net_thermal_radiation_sum': 'Net surface thermal radiation',
            'wind_speed_10m': '10 m wind speed',
            'PM25': 'PM2.5',
            'NTL': 'Nighttime lights intensity',
            'NDBI': 'Normalized difference built-up index',
            'EVI': 'Enhanced vegetation index',
            'ACS_PCT_BLACK_NONHISP': '% of African American (non-Hispanic)',
            'ACS_PCT_HISPANIC': '% of Hispanic',
            'ACS_PCT_AGE_ABOVE65': '% of age >= 65',
            'ACS_PCT_ENGL_NOT_ALL': '% who do not speak English at all',
            'ACS_PCT_PERSON_INC_BELOW99': '% of income below 100% poverty line',
            'ACS_PCT_UNEMPLOY': '% unemployed',
            'AC_RECS': '% of AC adoption',
            'ACS_PCT_HU_ELEC': '% of housing units heating with electricity',
            'ACS_PCT_HU_SOLAR': '% of housing units heating with solar',
            'ACS_PCT_OWNER_HU_COST_30PCT': '% of owners with housing costs >= 30% of income',
            'ACS_PCT_RENTER_HU': '% of renters',
            'ACS_PCT_1UP_PERS_1ROOM': '% of occupied housing units with >1 person/room',
            'ACS_MEDIAN_HOME_VALUE': 'Median home value',
            'ACS_MEDIAN_YEAR_BUILT': 'Median year built',
            'PCT_DEM': '% of Democratic votes',
            'RH_mean': 'Mean humidity'},
 'target': 'EB_pct',
 'seed': 3407,
 'period': '2009–2022',
 'seasons': {'SP': 'Spring', 'SU': 'Summer', 'FA': 'Fall', 'WI': 'Winter', 'ALL': 'Overall'},
 'xgb_params': None,
 'gxgb_params': {'n_estimators': 500,
                 'max_depth': 6,
                 'learning_rate': 0.05,
                 'subsample': 0.8,
                 'colsample_bytree': 0.8,
                 'colsample_bylevel': 0.8,
                 'objective': 'reg:squarederror',
                 'random_state': 3407,
                 'reg_alpha': 0.1,
                 'reg_lambda': 1.0,
                 'min_child_weight': 1,
                 'gamma': 0,
                 'tree_method': 'approx',
                 'n_jobs': 1,
                 'verbosity': 0},
 'bandwidths': {},
 'kernel': 'adaptive_bisquare',
 'distance': 'haversine_km',
 'earth_radius_km': 6371.0088,
 'local_weight': 1.0,
 'pdp_bootstrap_replicates': 1000,
 'xgb_base_params': {'learning_rate': 0.05,
                     'subsample': 0.8,
                     'colsample_bytree': 0.8,
                     'colsample_bylevel': 0.8,
                     'gamma': 0,
                     'random_state': 3407,
                     'tree_method': 'hist',
                     'n_jobs': 1,
                     'objective': 'reg:squarederror',
                     'verbosity': 0}}
FEATURES = CFG["features"]
LABELS = CFG["labels"]
SEASONS = ["SP", "SU", "FA", "WI"]
MODELS = ["ALL", *SEASONS]
SEASON_NAMES = CFG["seasons"]
SEED = CFG["seed"]
SAMPLE_FILES = {"SP": "spring_sample.csv", "SU": "summer_sample.csv", "FA": "fall_sample.csv", "WI": "winter_sample.csv"}
for name in ["figures", "tables"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)


In [ ]:
def load_data(code):
    if code == "ALL":
        frames = [load_data(s)[["GEOID", *FEATURES, "EB_pct"]] for s in SEASONS]
        frame = pd.concat(frames, ignore_index=True).groupby("GEOID", sort=True)[FEATURES + ["EB_pct"]].mean().reset_index()
    else:
        frame = pd.read_csv(DATA / SAMPLE_FILES[code], dtype={"GEOID": str}, float_precision="round_trip")
        frame["GEOID"] = frame["GEOID"].str.zfill(5)
        frame = frame.sort_values("GEOID").reset_index(drop=True)
    frame["state"] = frame["GEOID"].str[:2]
    assert frame["GEOID"].is_unique and len(frame) == 2830
    assert np.isfinite(frame[FEATURES + ["EB_pct"]].to_numpy()).all()
    return frame

def split_indices(n):
    return train_test_split(np.arange(n), test_size=0.20, random_state=SEED)

def load_geo(layer="counties"):
    import geopandas as gpd
    location = COUNTY_FILE if layer == "counties" else STATE_FILE
    frame = gpd.read_file(location)
    needed = ["GEOID", "STATEFP"] if layer == "counties" else ["STATEFP", "STUSPS"]
    frame["STATEFP"] = frame["STATEFP"].astype(str).str.zfill(2)
    if layer == "counties":
        frame["GEOID"] = frame["GEOID"].astype(str).str.zfill(5)
    return frame[needed + ["geometry"]].to_crs("EPSG:4326")


In [ ]:
XGB_VARIANTS = [
    ("depth6_500", 500, 6, 0.10, 1.00, 1),
    ("depth4_800", 800, 4, 0.10, 1.00, 1),
    ("depth3_1000", 1000, 3, 0.10, 1.00, 1),
    ("depth4_regularized", 800, 4, 0.50, 5.00, 5),
    ("depth2_1500", 1500, 2, 0.10, 1.00, 1),
    ("depth5_regularized", 800, 5, 0.50, 5.00, 5),
]

def xgb_candidates():
    return {
        name: dict(CFG["xgb_base_params"], n_estimators=trees, max_depth=depth,
                   reg_alpha=alpha, reg_lambda=regularization, min_child_weight=child)
        for name, trees, depth, alpha, regularization, child in XGB_VARIANTS
    }

_XGB_SELECTION = globals().get("_XGB_SELECTION")

def select_xgb_parameters(force=False):
    global _XGB_SELECTION
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import root_mean_squared_error

    if force or _XGB_SELECTION is None:
        variants = list(XGB_VARIANTS)
        settings = xgb_candidates()
        configurations = pd.DataFrame([
            dict(configuration=i + 1, param=name, **settings[name])
            for i, (name, *_) in enumerate(variants)
        ])
        data = {season: load_data(season) for season in SEASONS}
        train, test = split_indices(len(data["SP"]))
        folds = [(train[a], train[b]) for a, b in GroupKFold(5).split(
            train, groups=data["SP"].iloc[train]["GEOID"].str[:2])]
        assert all(set(a).isdisjoint(test) and set(b).isdisjoint(test) for a, b in folds)
        assert all(set(data["SP"].iloc[a]["GEOID"].str[:2]).isdisjoint(
            data["SP"].iloc[b]["GEOID"].str[:2]) for a, b in folds)
        records = []
        for number, (name, *_) in enumerate(variants, start=1):
            for season in SEASONS:
                table = data[season]
                for fold, (training, validation) in enumerate(folds):
                    fit = table.iloc[training]
                    held_out = table.iloc[validation]
                    model = XGBRegressor(**settings[name])
                    model.fit(fit[FEATURES], fit["EB_pct"])
                    prediction = model.predict(held_out[FEATURES])
                    rmse = root_mean_squared_error(held_out["EB_pct"], prediction)
                    scale = fit["EB_pct"].std(ddof=1)
                    records.append({
                        "configuration": number, "param": name, "season": season,
                        "fold": fold, "RMSE_pp": rmse, "nRMSE": rmse / scale,
                        "train_target_SD": scale, "n_train": len(fit), "n_test": len(held_out)
                    })
            print(name, "completed", flush=True)
        scores = pd.DataFrame(records)
        assert len(scores) == 120
        assert scores.groupby("param").size().eq(20).all()
        fold_means = scores.groupby(["configuration", "param", "fold"], as_index=False)["nRMSE"].mean()
        summary = fold_means.groupby(["configuration", "param"])["nRMSE"].agg(
            ["mean", "std", "count"]).reset_index()
        summary["SE"] = summary["std"] / np.sqrt(summary["count"])
        winner = summary["mean"].idxmin()
        summary["selected"] = summary.index == winner
        selected_name = summary.loc[winner, "param"]
        results = configurations.merge(summary, on=["configuration", "param"], validate="1:1").sort_values("configuration")
        _XGB_SELECTION = {
            "configuration": int(summary.loc[winner, "configuration"]),
            "parameters": dict(settings[selected_name]),
            "scores": scores, "summary": summary, "results": results
        }
    CFG["xgb_params"] = dict(_XGB_SELECTION["parameters"])
    return dict(CFG["xgb_params"])

def fit_global(code):
    parameters = select_xgb_parameters()
    data = load_data(code)
    train_indices, test_indices = split_indices(len(data))
    model = XGBRegressor(**parameters)
    model.fit(data.iloc[train_indices][FEATURES], data.iloc[train_indices]["EB_pct"])
    return {
        "data": data,
        "model": model,
        "train_indices": train_indices,
        "test_indices": test_indices,
        "prediction": model.predict(data[FEATURES])
    }


def export_table(stem, table):
    table.to_csv(OUT / "tables" / f"{stem}.csv", index=False, float_format="%.2f")


In [ ]:
STATE_REGIONS = {
    state: region
    for region, states in {
        "Northeast": "09 23 25 33 44 50 34 36 42",
        "Central": "17 18 26 39 55 19 20 27 29 31 38 46",
        "South": "10 11 12 13 24 37 45 51 54 01 21 28 47 05 22 40 48",
        "West": "04 08 16 30 32 35 49 56 02 06 15 41 53",
    }.items()
    for state in states.split()
}

def county_regions(geoids):
    states = geoids.astype(str).str.zfill(5).str[:2]
    regions = states.map(STATE_REGIONS)
    if regions.isna().any():
        raise ValueError("Unmapped state FIPS codes: " + ", ".join(sorted(states[regions.isna()].unique())))
    return regions
